# 01 · FLUX.2 [klein] Locally: Generation & Parameter Ablations

**Hardware**: 🟡 consumer GPU (the 4B model is ~13GB VRAM in bf16; less with `enable_model_cpu_offload`. Apple Silicon mps works, slower)

## What you will learn

1. Run the Apache-2.0 FLUX.2 [klein] — the early-2026 flagship of fast open-weight image generation
2. **Ablate the three core parameters** — steps, guidance_scale, seed — and build parameter-to-picture intuition
3. How the base build (full CFG training) relates to the distilled build (cf. step distillation in [theory.md](../theory.md) §3)
4. Visualize flow-matching sampling: watch noise become an image step by step

> Version note (2026-08): defer to the [FLUX.2-klein-4B model card](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B); first run requires accepting the license and `huggingface-cli login`.

In [ ]:
%pip install -q "diffusers>=0.36" transformers accelerate safetensors torch matplotlib

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# base build: no guidance distillation — parameters behave "textbook", ideal for ablations
# for speed, switch to "black-forest-labs/FLUX.2-klein-4B" (distilled, few-step)
pipe = Flux2KleinPipeline.from_pretrained(
    "black-forest-labs/FLUX.2-klein-base-4B", torch_dtype=torch.bfloat16
)
if device == "cuda":
    pipe.enable_model_cpu_offload()   # the VRAM lifesaver
else:
    pipe = pipe.to(device)

## 1. First image

In [ ]:
prompt = (
    "A cozy bookstore cafe at dusk, warm window light, a cat sleeping on a stack of books, "
    "a chalkboard sign that says 'Multimodal 101', photorealistic, 35mm"
)

image = pipe(
    prompt=prompt,
    height=768, width=768,
    guidance_scale=4.0,
    num_inference_steps=28,
    generator=torch.Generator(device="cpu").manual_seed(42),
).images[0]
image.save("first.png")
image

Check two things: **text rendering** (is 'Multimodal 101' spelled right on the sign?) and **lighting coherence** (is the window light consistent in direction?) — the two most visible ways this generation leaves the SD1.5 era behind.

## 2. Ablation 1: sampling steps

Flow matching learns a noise-to-image velocity field; steps = the precision of integrating along that trajectory. Find where quality saturates — every step beyond that is wasted money.

In [ ]:
import matplotlib.pyplot as plt
import time

steps_list = [4, 8, 16, 28, 50]
fig, axes = plt.subplots(1, len(steps_list), figsize=(20, 4.5))
for ax, steps in zip(axes, steps_list):
    t0 = time.perf_counter()
    img = pipe(prompt=prompt, height=512, width=512, guidance_scale=4.0,
               num_inference_steps=steps,
               generator=torch.Generator(device="cpu").manual_seed(42)).images[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{steps} steps / {time.perf_counter()-t0:.1f}s")
plt.tight_layout(); plt.show()

# The base build should clearly degrade at 4-8 steps — while the distilled klein
# produces good images at 4. That gap IS the value of step distillation (theory.md §3).

## 3. Ablation 2: guidance scale

CFG strength = "how hard to obey the prompt". Too low drifts off-topic; too high oversaturates and stiffens composition.

In [ ]:
cfg_list = [1.0, 2.5, 4.0, 7.0, 12.0]
fig, axes = plt.subplots(1, len(cfg_list), figsize=(20, 4.5))
for ax, cfg in zip(axes, cfg_list):
    img = pipe(prompt=prompt, height=512, width=512, guidance_scale=cfg,
               num_inference_steps=28,
               generator=torch.Generator(device="cpu").manual_seed(42)).images[0]
    ax.imshow(img); ax.axis("off"); ax.set_title(f"cfg={cfg}")
plt.tight_layout(); plt.show()

## 4. Ablation 3: seeds and the "gacha" workflow

Same prompt, different seeds = different samples from one distribution. "Fix the seed, iterate the prompt" is the fundamental move for controlled iteration in production.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, seed in zip(axes, [0, 1, 2, 3]):
    img = pipe(prompt=prompt, height=512, width=512, guidance_scale=4.0,
               num_inference_steps=28,
               generator=torch.Generator(device="cpu").manual_seed(seed)).images[0]
    ax.imshow(img); ax.axis("off"); ax.set_title(f"seed={seed}")
plt.tight_layout(); plt.show()

## Exercises

1. **Chinese text-rendering stress test**: put "多模态101" on the sign and compare against [Qwen-Image](../landscape.md) — Chinese rendering is its home turf.
2. A GenEval-style compositionality test: "a red cube on top of a blue sphere, to the left of a yellow cone" — count what lands correctly.
3. Send the same prompts to the GPT Image 2 / Nano Banana APIs and run an open-vs-closed blind vote with friends.
4. Try img2img: `pipe(image=..., strength=0.6, ...)` to restyle your own photo — a warm-up for the editing half of this chapter.